In [23]:
from datasets import load_dataset, DatasetDict, Audio, Dataset, concatenate_datasets
from datasets import Value
import pandas as pd
import os

In [28]:
def load_local_dataset(PATH):
    
    # Chemins
    base_dir = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets"
    PATH_TO_AUDIO = os.path.join(base_dir, PATH, "clips")

    # Lire les TSV
    df_train = pd.read_csv(os.path.join(base_dir, PATH, "train.tsv"), delimiter='\t')
    df_dev   = pd.read_csv(os.path.join(base_dir, PATH, "dev.tsv"), delimiter='\t')
    df_test  = pd.read_csv(os.path.join(base_dir, PATH, "test.tsv"), delimiter='\t')

    # Filtrage sans apply
    def filter_files(df):
        filtered_rows = []
        for _, row in df.iterrows():
            audio_file = os.path.join(PATH_TO_AUDIO, row['path'])
            if os.path.isfile(audio_file):
                filtered_rows.append(row)
        return pd.DataFrame(filtered_rows)

    df_train = filter_files(df_train)
    df_dev   = filter_files(df_dev)
    df_test  = filter_files(df_test)

    # Convertir en Dataset HuggingFace
    common_voice = DatasetDict({
        "train": Dataset.from_pandas(df_train),
        "dev": Dataset.from_pandas(df_dev),
        "test": Dataset.from_pandas(df_test),
    })

    for split, dataset in common_voice.items():
        print(f"\n=== SPLIT : {split} ===")
        print(f"Nombre d'exemples : {len(dataset)}")
    
    # Ajouter colonne audio complète
    for split in ["train", "dev", "test"]:
        common_voice[split] = common_voice[split].map(
            lambda x: {"audio": os.path.join(PATH_TO_AUDIO, x["path"])},
            batched=False
        )

    # Reechantillonnage audio
    for split in ["train", "dev", "test"]:
        common_voice[split] = common_voice[split].cast_column("audio", Audio(sampling_rate=16_000))

    COLUMNS_TO_CAST_AS_STRING = [
        "gender",
        "age",
        "accents",
        "variant",
        "segment"
    ]

    for split in ["train", "dev", "test"]:
        for col in COLUMNS_TO_CAST_AS_STRING:
            if col in common_voice[split].column_names:
                common_voice[split] = common_voice[split].cast_column(
                    col, Value("string")
                )

    common_voice["train"] = concatenate_datasets([common_voice["train"], common_voice["dev"]])
    split = common_voice["test"].train_test_split(
        test_size=100,
        shuffle=True,
        seed=42
    )

    common_voice["test"] = split["test"]        # 100 exemples
    common_voice["train"] = concatenate_datasets([
    common_voice["train"], split["train"]])

    del common_voice["dev"]

    return common_voice

In [29]:
path = 'zazaki/'

In [30]:
common_voice = load_local_dataset(path)


=== SPLIT : train ===
Nombre d'exemples : 734

=== SPLIT : dev ===
Nombre d'exemples : 463

=== SPLIT : test ===
Nombre d'exemples : 392


Casting the dataset: 100%|██████████| 392/392 [00:00<00:00, 67569.44 examples/s]


In [16]:
common_voice['train']

Dataset({
    features: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', '__index_level_0__', 'audio'],
    num_rows: 1197
})

In [31]:
common_voice

DatasetDict({
    train: Dataset({
        features: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', '__index_level_0__', 'audio'],
        num_rows: 1489
    })
    test: Dataset({
        features: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', '__index_level_0__', 'audio'],
        num_rows: 100
    })
})

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join("..", "results","whisperBaseKurdZazaki"),  # repertoire enregistrement des chekcpoints
    do_train=True,                      
    do_eval=True,  
    per_device_train_batch_size=4,        # petit batch par GPU
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,        # pour batch effectif ~32
    learning_rate=1e-5,
    warmup_steps=50,                      # warmup pour stabiliser
    max_steps=230,                        # 5 epochs complètes
    lr_scheduler_type="cosine",           # scheduler cosinus
    gradient_checkpointing=False,         # économise la mémoire
    fp16=True,                            # demi-précision
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    remove_unused_columns=False
)


In [33]:
from transformers import WhisperFeatureExtractor, WhisperProcessor, WhisperForConditionalGeneration, WhisperTokenizer

In [40]:
model_dir = "/info/raid-etu/m1/s2506992/projet-m1-asr/results/whisperBaseKurd/checkpoint-750"

In [43]:
# Processor et tokenizer depuis le modèle de base
processor = WhisperProcessor.from_pretrained("openai/whisper-base", language="turkish", task="transcribe")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-base", language="turkish", task="transcribe")

# Feature extractor (optionnel si besoin direct)
feature_extractor = processor.feature_extractor

# Charger le modèle fine-tuné depuis ton checkpoint local
model = WhisperForConditionalGeneration.from_pretrained(model_dir)

In [44]:
from transformers import WhisperFeatureExtractor, WhisperProcessor, WhisperForConditionalGeneration, WhisperTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
from datasets import load_dataset, DatasetDict, Audio, Dataset, concatenate_datasets
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import pandas as pd
import evaluate
import torch
import os


"""
NB: La methode load_dataset de huggingface ne marche plus pour les nouveaux datasets de common voice, Alors on telecharge le dataset
en local, on le convertis en dataframe pandas, puis on le convertit en dataset huggingface et on fait le reechantillonage des datasets 
"""
def load_local_dataset(PATH):
    
    # Chemins
    base_dir = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets"
    PATH_TO_AUDIO = os.path.join(base_dir, PATH, "clips")

    # Lire les TSV
    df_train = pd.read_csv(os.path.join(base_dir, PATH, "train.tsv"), delimiter='\t')
    df_dev   = pd.read_csv(os.path.join(base_dir, PATH, "dev.tsv"), delimiter='\t')
    df_test  = pd.read_csv(os.path.join(base_dir, PATH, "test.tsv"), delimiter='\t')

    # Filtrage sans apply
    def filter_files(df):
        filtered_rows = []
        for _, row in df.iterrows():
            audio_file = os.path.join(PATH_TO_AUDIO, row['path'])
            if os.path.isfile(audio_file):
                filtered_rows.append(row)
        return pd.DataFrame(filtered_rows)

    df_train = filter_files(df_train)
    df_dev   = filter_files(df_dev)
    df_test  = filter_files(df_test)

    # Convertir en Dataset HuggingFace
    common_voice = DatasetDict({
        "train": Dataset.from_pandas(df_train),
        "dev": Dataset.from_pandas(df_dev),
        "test": Dataset.from_pandas(df_test),
    })
    
    # Ajouter colonne audio complète
    for split in ["train", "dev", "test"]:
        common_voice[split] = common_voice[split].map(
            lambda x: {"audio": os.path.join(PATH_TO_AUDIO, x["path"])},
            batched=False
        )

    # Reechantillonnage audio
    for split in ["train", "dev", "test"]:
        common_voice[split] = common_voice[split].cast_column("audio", Audio(sampling_rate=16_000))

    COLUMNS_TO_CAST_AS_STRING = [
        "gender",
        "age",
        "accents",
        "variant",
        "segment"
    ]

    for split in ["train", "dev", "test"]:
        for col in COLUMNS_TO_CAST_AS_STRING:
            if col in common_voice[split].column_names:
                common_voice[split] = common_voice[split].cast_column(
                    col, Value("string")
                )

    common_voice["train"] = concatenate_datasets([common_voice["train"], common_voice["dev"]])
    split = common_voice["test"].train_test_split(
        test_size=100,
        shuffle=True,
        seed=42
    )

    common_voice["test"] = split["test"]        # 100 exemples
    common_voice["train"] = concatenate_datasets([
    common_voice["train"], split["train"]])

    del common_voice["dev"]

    return common_voice
"""
cette methode sert a creer deux nouvelles colonnes, une colonne: inputfeatures; elle contient un tableau
numpy qui represente les caracteres acoustiques de l'audio, et une colonne label; elle contient la label dans le format Token
"""
def prepare_dataset(batch, tokenizer):
    
    # on transforme le text en token numerique
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    
    return batch

"""
Un Data Collator sert a preparer le bacth avant de l'envoyer au model, dans notre cas il va servir pour faire 3 choses:
- etape 1: on transforme la colonne 'input features' qui contient les tableaux numpy en tensor pytorch
- etape 2: on unifie la taille des tokens numerique (qui represente les labels) en ajoutant des <pad> qui seront apres remplace par des -100 pour ne pas biaiser 
  les metrics d'evaluation
- etape 3: on supprime le token <s> qui indique le debut de sequence
"""

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # décodage audio batch-wise
        audio_arrays = [f["audio"]["array"] for f in features]
        sampling_rate = features[0]["audio"]["sampling_rate"]

        inputs = self.processor.feature_extractor(
            audio_arrays,
            sampling_rate=sampling_rate,
            return_tensors="pt"
        )

        # traitement labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # retirer le token <s> au début si présent
        if (labels[:, 0] == self.decoder_start_token_id).all():
            labels = labels[:, 1:]

        # assigner correctement
        inputs["labels"] = labels

        return inputs

        
# Charger la  metric
metric = evaluate.load("wer")

# Defintion des Hyperparametre de l'entrainement
training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join("..", "results","whisperBaseKurdZazaki"),  # repertoire enregistrement des chekcpoints
    do_train=True,                      
    do_eval=True,  
    per_device_train_batch_size=4,        # petit batch par GPU
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,        # pour batch effectif ~32
    learning_rate=1e-5,
    warmup_steps=50,                      # warmup pour stabiliser
    max_steps=230,                        # 5 epochs complètes
    lr_scheduler_type="cosine",           # scheduler cosinus
    gradient_checkpointing=False,         # économise la mémoire
    fp16=True,                            # demi-précision
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    remove_unused_columns=False
)

def whisper_finetune (PATH):

    model_dir = "/info/raid-etu/m1/s2506992/projet-m1-asr/results/whisperBaseKurd/checkpoint-750"
    
    processor = WhisperProcessor.from_pretrained("openai/whisper-base", language="turkish", task="transcribe")
    tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-base", language="turkish", task="transcribe")

    # Feature extractor t)
    feature_extractor = processor.feature_extractor

    # Charger le model finetunner sur le kurd
    model = WhisperForConditionalGeneration.from_pretrained(model_dir)

    # on recupere le dataset
    common_voice = load_local_dataset(PATH)
    
    # le dataset avec deux colonnes, input_features: spectogramme mel et labels: les tokens numerique des etiquetes
    common_voice = common_voice.map(
    prepare_dataset,
    fn_kwargs={"tokenizer": tokenizer},
    remove_columns=[],              
    num_proc=1,
    load_from_cache_file=False
    )
    
    # Initialiser le datacollator
    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
    )

    """
    elle prends les predictions, elle les compare les predictions et les labels, et retourne la word_error_rate
    """
    def compute_metrics(pred):

        normalizer = BasicTextNormalizer()
    
        # on recupere les predictions
        pred_ids = pred.predictions
    
        # on recupere les labels 
        label_ids = pred.label_ids

        # replace -100 with the pad_token_id
        label_ids[label_ids == -100] = tokenizer.pad_token_id

        # on decode (on convertit les tokens numerique en text) les deux, pred_ids et label_ids
        pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        pred_str = [normalizer(s) for s in pred_str]
        label_str = [normalizer(s) for s in label_str]

        # calcul wer
        wer = 100 * metric.compute(predictions=pred_str, references=label_str)

        # calcul cer
        cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)

        return {"wer": wer, "cer": cer}  

    # Initialiser le trainer
    trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    )
    
    # lancer l'entrainement
    trainer.train()
    
if __name__ == "__main__":
    PATH = 'zazaki/'
    whisper_finetune(PATH)

Map (num_proc=1): 100%|██████████| 100/100 [00:00<00:00, 100.96 examples/s]
/tmp/ipykernel_447398/364932161.py:228: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/info/etu/m1/s2506992/miniconda3/envs/asr2526_new/lib/python3.10/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


KeyboardInterrupt: 